# 02: Interactive Unstructured Geodesic Mesh (UGRID NetCDF)

This notebook demonstrates **interactive 3D spherical visualization** of an unstructured triangular geodesic mesh:
1. Loading node coordinates, face connectivity, and scalar fields from a UGRID-1.0 NetCDF file (`data/unstructured/unstructured_geodesic_mesh.nc`).
2. Constructing an unstructured spherical 3D mesh with `geovista.Transform.from_unstructured`.
3. Highlighting triangular cell wireframe edges (`show_edges=True`, `edge_color="navy"`).
4. Interactively rotating and zooming into individual geodesic cells on the 3D globe with `map_single_frame(..., sFilename_out=None)`.

In [ ]:
import os
from pathlib import Path
import netCDF4 as nc
import numpy as np
import geovista as gv

import pyvista as pv

# Configure interactive 3D rendering backend in Jupyter:
# "html" embeds the complete interactive 3D WebGL scene inline (srcdoc),
# preventing VS Code / browser iframe security from blocking localhost ports.
try:
    import trame

    pv.set_jupyter_backend("html")
    print("✓ Interactive backend: in-cell 3D WebGL widget (html)")
except ImportError:
    pv.set_jupyter_backend("none")
    print("ℹ Interactive backend: native desktop 3D window")
    print("  To embed interactive 3D widgets directly inside notebook cells: pip install 'pyvista[jupyter]'")

import pyearthviz3d
from pyearthviz3d import (
    VisualizationConfig,
    ScalarBarConfig,
    map_single_frame,
)

# Resolve repository root and data path
REPO_ROOT = Path(os.getcwd()).resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

UGRID_PATH = REPO_ROOT / "data" / "unstructured" / "unstructured_geodesic_mesh.nc"
print(f"pyearthviz3d version: {pyearthviz3d.__version__}")
print(f"Target UGRID dataset: {UGRID_PATH} (exists: {UGRID_PATH.exists()})")

## Step 1: Load and Inspect the UGRID NetCDF Dataset

Unstructured climate and ocean models (such as MPAS, ICON, and SE-HOMME) store mesh coordinates as 1D arrays of nodes along with a 2D connectivity table specifying node indices for each triangular or polygonal face.

In [ ]:
with nc.Dataset(UGRID_PATH, "r") as ds:
    node_lons = ds.variables["Mesh2_node_x"][:]
    node_lats = ds.variables["Mesh2_node_y"][:]
    face_nodes = ds.variables["Mesh2_face_nodes"][:]
    vorticity = ds.variables["vorticity"][:]
    ssh = ds.variables["sea_surface_height"][:]

print(f"Number of nodes: {len(node_lons)}")
print(f"Number of triangular faces: {len(face_nodes)}")
print(f"Face connectivity shape: {face_nodes.shape} (3 nodes per face)")
print(f"Vorticity range: [{vorticity.min():.2e}, {vorticity.max():.2e}] 1/s")
print(f"SSH range:       [{ssh.min():.2f}, {ssh.max():.2f}] m")

## Step 2: Build Unstructured 3D Spherical Mesh

Using `geovista.Transform.from_unstructured`, we assemble the spherical polyhedron by projecting nodes onto the sphere and connecting triangular cells according to `face_nodes`.

In [ ]:
mesh = gv.Transform.from_unstructured(
    xs=node_lons,
    ys=node_lats,
    connectivity=face_nodes,
    data=vorticity,
    name="vorticity",
    start_index=0,
)
mesh.cell_data["sea_surface_height"] = ssh
valid_indices = np.arange(mesh.n_cells)

print(f"Constructed PyVista PolyData: {mesh.n_cells} cells, {mesh.n_points} vertices.")

## Step 3: Interactive 3D Visualization of Vorticity with Wireframe Edges

Passing `sFilename_out=None` launches the interactive 3D view.
Notice how individual triangular geodesic cells and their edges (`edge_color="navy"`) can be inspected closely by zooming in with the mouse wheel.

In [ ]:
viz_config = VisualizationConfig(
    longitude_focus=-30.0,  # North Atlantic vantage
    latitude_focus=40.0,
    zoom_factor=0.85,
    show_coastlines=True,
    show_graticule=True,
    coastline_color="black",
    coastline_width=1.5,
    colormap="coolwarm",
    verbose=True,
)

scalar_config = ScalarBarConfig(
    title="Relative Vorticity",
    title_font_size=12,
    label_font_size=10,
    fmt="%.1e",
    n_labels=5,
    orientation="horizontal",
)

# Launch interactive 3D view
result = map_single_frame(
    pMesh=mesh,
    aValid_cell_indices=valid_indices,
    pConfig=viz_config,
    sScalar="vorticity",
    sUnit="1/s",
    sFilename_out=None,  # Interactive 3D mode
    scalar_config=scalar_config,
    show_edges=True,
    edge_color="navy",
    return_detailed_result=True,
)

if hasattr(result, "get_summary"):
    print(result.get_summary())

## Step 4: Interactive Sea Surface Height (SSH) Visualization

Now let's explore Sea Surface Height (SSH) anomalies across the global ocean basins interactively.

In [ ]:
viz_ssh_config = VisualizationConfig(
    longitude_focus=150.0,  # Pacific Ocean focus
    latitude_focus=10.0,
    zoom_factor=0.85,
    show_coastlines=True,
    show_graticule=True,
    coastline_color="white",
    coastline_width=1.2,
    colormap="viridis",
    verbose=True,
)

scalar_ssh_config = ScalarBarConfig(
    title="Sea Surface Height Anomaly",
    title_font_size=12,
    label_font_size=10,
    fmt="%.2f",
    n_labels=6,
    orientation="horizontal",
)

# Launch interactive 3D view
map_single_frame(
    pMesh=mesh,
    aValid_cell_indices=valid_indices,
    pConfig=viz_ssh_config,
    sScalar="sea_surface_height",
    sUnit="m",
    sFilename_out=None,  # Interactive 3D mode
    scalar_config=scalar_ssh_config,
    show_edges=True,
    edge_color="gray",
)

## Optional: Static PNG Export

In [ ]:
from IPython.display import Image, display

output_png = "unstructured_mesh_export.png"
map_single_frame(
    pMesh=mesh,
    aValid_cell_indices=valid_indices,
    pConfig=viz_config,
    sScalar="vorticity",
    sUnit="1/s",
    sFilename_out=output_png,
    scalar_config=scalar_config,
    show_edges=True,
    edge_color="navy",
)
display(Image(output_png, width=600))